<a href="https://colab.research.google.com/github/NAMUORI00/aerospace-rag/blob/main/notebooks/aerospace_rag_colab_ui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aerospace Local RAG Notebook UI

현재 `data/` 폴더의 항공우주 파일 5개를 Qdrant local mode, FalkorDB/FalkorDBLite graph channel, BM25 keyword channel로 인덱싱하고 질의합니다. LLM 호출은 vLLM 대신 Ollama `gemma4:e2b`를 사용합니다.

## 1. Colab/로컬 프로젝트 준비

Colab에서는 Google Drive를 전혀 mount하거나 재사용하지 않습니다. 실행할 때마다 `/content`로 이동해 기존 `/content/aerospace-rag`를 삭제하고 `https://github.com/NAMUORI00/aerospace-rag.git`를 새로 clone합니다. 로컬 실행에서는 현재/상위 폴더에서 프로젝트 루트를 찾습니다.

In [ ]:
from pathlib import Path
import importlib.util
import os
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request

try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

GITHUB_REPO_URL = "https://github.com/NAMUORI00/aerospace-rag.git"
DEFAULT_COLAB_ROOT = Path("/content/aerospace-rag")


def ensure_valid_cwd() -> Path:
    try:
        return Path.cwd()
    except FileNotFoundError:
        fallback = DEFAULT_COLAB_ROOT.parent if DEFAULT_COLAB_ROOT.parent.exists() else Path.home()
        os.chdir(fallback)
        return Path.cwd()


def project_root_candidates() -> list[Path]:
    cwd = ensure_valid_cwd()
    return [cwd, cwd.parent]


def is_project_root(path: Path) -> bool:
    return (path / "aerospace_rag").exists() and (path / "data").exists()


def find_project_root() -> Path | None:
    for candidate in project_root_candidates():
        if is_project_root(candidate):
            return candidate
    return None


PROJECT_ROOT = None

if IN_COLAB:
    os.chdir(DEFAULT_COLAB_ROOT.parent)
    print("Cloning project from GitHub:", GITHUB_REPO_URL)
    if DEFAULT_COLAB_ROOT.exists():
        shutil.rmtree(DEFAULT_COLAB_ROOT)
    subprocess.check_call(["git", "clone", GITHUB_REPO_URL, str(DEFAULT_COLAB_ROOT)])
    PROJECT_ROOT = DEFAULT_COLAB_ROOT if is_project_root(DEFAULT_COLAB_ROOT) else None
else:
    PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. Colab git clone 또는 로컬 프로젝트 경로를 확인하세요."
    )

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("In Colab:", IN_COLAB)
print("Repo URL:", GITHUB_REPO_URL)

## 2. 의존성 설치

Colab 런타임은 매번 새로 시작될 수 있으므로 필요한 패키지를 확인하고 설치합니다. Ollama는 별도 셀에서 설치/실행합니다.

In [ ]:
def ensure_dependencies() -> None:
    required = {
        "qdrant_client": "qdrant-client",
        "falkordb": "falkordb",
        "openpyxl": "openpyxl",
        "pypdf": "pypdf",
        "ipywidgets": "ipywidgets",
        "nbformat": "nbformat",
    }
    missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
    if missing:
        print("Installing:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Base dependencies already installed")

    if importlib.util.find_spec("redislite") is None and IN_COLAB:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "falkordblite>=0.9"])
            print("FalkorDBLite installed")
        except Exception as exc:
            print("FalkorDBLite optional install skipped:", exc)


ensure_dependencies()

## 3. Ollama 준비

Colab에서는 Ollama를 설치하고 서버를 백그라운드로 실행한 뒤 `gemma4:e2b`를 pull합니다. 로컬에서는 Ollama 앱/서버를 미리 실행해 두면 됩니다.

In [ ]:
os.environ.setdefault("LLM_PROVIDER", "ollama")
os.environ.setdefault("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
os.environ.setdefault("OLLAMA_MODEL", "gemma4:e2b")


def ollama_api_ok() -> bool:
    try:
        with urllib.request.urlopen(os.environ["OLLAMA_BASE_URL"].rstrip("/") + "/api/tags", timeout=5) as response:
            return response.status == 200
    except Exception:
        return False


def ensure_ollama_runtime(pull_model: bool = True) -> None:
    model = os.environ["OLLAMA_MODEL"]
    if not IN_COLAB:
        print("Local runtime: ensure Ollama is running separately.")
        print("Expected:", os.environ["OLLAMA_BASE_URL"], "model:", model)
        return

    if shutil.which("ollama") is None:
        print("Installing Ollama...")
        subprocess.check_call("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

    if not ollama_api_ok():
        print("Starting Ollama server...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
        for _ in range(60):
            if ollama_api_ok():
                break
            time.sleep(1)

    if not ollama_api_ok():
        raise RuntimeError("Ollama server did not become ready on " + os.environ["OLLAMA_BASE_URL"])

    if pull_model:
        print("Pulling Ollama model:", model)
        subprocess.check_call(["ollama", "pull", model])

    print("Ollama ready:", os.environ["OLLAMA_BASE_URL"], "model:", model)


ensure_ollama_runtime(pull_model=True)

## 4. 데이터 확인

v1은 아래 5개 파일만 사용합니다. Google Drive는 사용하지 않으며, Colab에서 파일이 없으면 브라우저 파일 업로드 창이 열립니다.

In [ ]:
from aerospace_rag.ingestion import EXPECTED_FILES

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
missing = [name for name in EXPECTED_FILES if not (DATA_DIR / name).exists()]

if missing and IN_COLAB and files is not None:
    print("누락된 데이터 파일:", missing)
    print("누락 파일을 업로드하세요. 파일명은 아래 목록과 정확히 같아야 합니다.")
    uploaded = files.upload()
    for filename in uploaded:
        source = Path(filename)
        if source.name in EXPECTED_FILES:
            shutil.move(str(source), DATA_DIR / source.name)

missing = [name for name in EXPECTED_FILES if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError("누락된 data 파일: " + ", ".join(missing))

for name in EXPECTED_FILES:
    print("OK", name)

## 5. 런타임 설정

기본값은 SmartFarm식 계약을 따르되, Colab smoke run이 안정적으로 끝나도록 임베딩은 hash fallback을 기본으로 둡니다. BGE-M3를 실제로 쓰려면 `requirements-models.txt`를 설치하고 `AEROSPACE_EMBED_BACKEND=sentence_transformers`로 바꾸세요.

In [ ]:
os.environ.setdefault("DAT_MODE", "hybrid")
os.environ.setdefault("AEROSPACE_EMBED_BACKEND", "hash")
print({
    "DAT_MODE": os.environ.get("DAT_MODE"),
    "LLM_PROVIDER": os.environ.get("LLM_PROVIDER"),
    "OLLAMA_MODEL": os.environ.get("OLLAMA_MODEL"),
    "AEROSPACE_EMBED_BACKEND": os.environ.get("AEROSPACE_EMBED_BACKEND"),
})

## 6. 인덱스 생성

PDF/XLSX/PNG 표를 청크화하고 Qdrant, BM25, FalkorDB graph helper 인덱스를 생성합니다.

In [ ]:
from aerospace_rag.pipeline import build_index

result = build_index(data_dir=DATA_DIR, index_dir=DATA_DIR / "index", reset=True)
print(result)

## 7. 질의 UI

`provider=None`이면 환경변수의 `LLM_PROVIDER=ollama`를 사용합니다. Ollama 서버가 준비되지 않았거나 모델 호출이 실패하면 자동으로 `extractive` 답변으로 fallback합니다.

In [ ]:
from aerospace_rag.pipeline import ask

question = "위성영상 가격은 저장영상과 신규촬영에서 어떻게 다른가?"
response = ask(question, index_dir=DATA_DIR / "index", top_k=5, provider=None, debug=True)
print(response.answer)
print("\nDiagnostics:", response.diagnostics)

## 8. 근거 확인

In [ ]:
for i, source in enumerate(response.sources, start=1):
    loc = f"p.{source.page}" if source.page else f"{source.sheet}:{source.row}" if source.row else "table"
    print(f"[{i}] {source.source_file} ({loc}) score={source.score:.3f} channels={source.channels}")
    print(source.excerpt[:500])
    print()

## 9. 다른 질문 예시

In [ ]:
sample_questions = [
    "H3 8호기 발사 실패 원인은?",
    "NASA solar sail 연구의 목적은?",
    "국가별 우주항공 예산과 인력 현황은?",
    "인공위성 판매대행사 선정 절차는 얼마나 걸리는가?",
]

for q in sample_questions:
    r = ask(q, index_dir=DATA_DIR / "index", top_k=4, provider=None, debug=True)
    print("QUESTION:", q)
    print(r.answer.splitlines()[0])
    print("sources:", [s.source_file for s in r.sources[:3]])
    print("-" * 80)